In [ ]:
!pip -q install pyarrow scikit-learn

from google.colab import drive
from pathlib import Path
from IPython.display import display

import gc
import unicodedata
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.neighbors import BallTree

drive.mount("/content/drive")

# 폐업 기록이 이 날짜까지 완전하게 수집됐다는 전제
OBSERVED_THROUGH = pd.Timestamp("2026-07-31")

BATCH_SIZE = 20_000
SMALL_N = 5
PRIOR_WEIGHT = 5
DAYS_PER_MONTH = 365.25 / 12
CLOSE_HISTORY_START = pd.Timestamp("2022-01-01")
EARTH_RADIUS_M = 6_371_008.8


def resolve_korean_path(path):
    target = Path(path)
    current = Path(target.anchor)

    for part in target.parts[1:]:
        direct = current / part

        if direct.exists():
            current = direct
            continue

        matches = [
            item for item in current.iterdir()
            if unicodedata.normalize("NFC", item.name)
            == unicodedata.normalize("NFC", part)
        ]

        if len(matches) != 1:
            raise FileNotFoundError(
                f"경로 확인 필요: {current / part}"
            )

        current = matches[0]

    return current


OLD_OUT = resolve_korean_path(
    "/content/drive/MyDrive/KB_FinalProject_Team6/"
    "완전쓰는거_ver2/인허가_파생지표_202301_202604"
)

OLD_SNAPSHOT_DIR = OLD_OUT / "snapshots"

NEW_OUT = (
    OLD_OUT.parent / "인허가_파생지표_202301_202608"
)

NEW_SNAPSHOT_DIR = NEW_OUT / "추가월별파일"
NEW_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# 이전 계산에서 사용했던 점포 기준표 재사용
d = pd.read_parquet(
    OLD_OUT / "store_master.parquet"
)

for col in ["open_s", "close_s"]:
    d[col] = pd.to_datetime(d[col], errors="raise")

if (
    d["점포이력ID"].isna().any()
    or d["점포이력ID"].duplicated().any()
):
    raise ValueError("점포 기준표의 점포번호 결측·중복을 확인하세요.")

# 좌표만 원본에서 읽기
master_path = resolve_korean_path(
    str(OLD_OUT.parent / "인허가_master.csv")
)

for encoding in ["utf-8-sig", "cp949"]:
    try:
        coords = pd.read_csv(
            master_path,
            encoding=encoding,
            usecols=["점포이력ID", "lon", "lat"],
            dtype="string",
        )
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("CSV 인코딩을 확인하세요.")

if (
    coords["점포이력ID"].isna().any()
    or coords["점포이력ID"].duplicated().any()
):
    raise ValueError("좌표 원본의 점포번호 결측·중복을 확인하세요.")

for col in ["lon", "lat"]:
    coords[col] = pd.to_numeric(
        coords[col], errors="coerce"
    ).astype(float)

coords["coord_valid"] = (
    np.isfinite(coords["lon"])
    & np.isfinite(coords["lat"])
    & coords["lon"].between(-180, 180)
    & coords["lat"].between(-90, 90)
    & ~(coords["lon"].eq(0) & coords["lat"].eq(0))
)

coords.loc[
    ~coords["coord_valid"],
    ["lon", "lat"],
] = np.nan

stores = d.merge(
    coords,
    on="점포이력ID",
    how="left",
    validate="one_to_one",
)

stores["coord_valid"] = (
    stores["coord_valid"].fillna(False).astype(bool)
)

del coords
gc.collect()

print(f"점포 기준표: {len(d):,}개")
print("기존 결과:", OLD_OUT)
print("새 결과:", NEW_OUT)

In [ ]:
def age_months(end, start):
    return (
        (end - start).dt.total_seconds()
        / (86400 * DAYS_PER_MONTH)
    )


def active(data, t):
    return (
        data["open_s"].lt(t)
        & (
            data["close_s"].isna()
            | data["close_s"].ge(t)
        )
    )


def lookup(rows, source, keys, value=None, how="size"):
    grouped = source.groupby(
        keys,
        observed=True,
        dropna=False,
    )

    result = (
        grouped.size()
        if value is None
        else grouped[value].agg(how)
    )

    index = pd.MultiIndex.from_frame(rows[keys])

    return pd.Series(
        result.reindex(index).to_numpy(),
        index=rows.index,
        dtype=float,
    )


def loo_median(rows, keys, value):
    result = pd.Series(np.nan, index=rows.index)

    for _, group in rows.groupby(
        keys,
        observed=True,
        sort=False,
    ):
        group = group.sort_values(value, kind="stable")
        values = group[value].to_numpy()
        n = len(values)

        if n < 2:
            continue

        ranks = np.arange(n)
        left = (n - 2) // 2
        right = (n - 1) // 2

        medians = (
            values[left + (ranks <= left)]
            + values[right + (ranks <= right)]
        ) / 2

        result.loc[group.index] = medians

    return result

In [ ]:
def build_month(d, t, observed_through=None):
    t6 = t - pd.DateOffset(months=6)
    t12 = t - pd.DateOffset(months=12)

    x = d.loc[active(d, t)].copy()

    x["base_ym"] = t.strftime("%Y-%m")
    x["age_m"] = age_months(t, x["open_s"])

    x["age_band"] = pd.cut(
        x["age_m"],
        bins=[0, 12, 18, 24, 36, np.inf],
        labels=["0–12", "12–18", "18–24", "24–36", "36+"],
        right=False,
    )

    x["is_new"] = x["age_m"].lt(12).astype("int8")
    x["biz_type"] = x["업종명"]

    x["dong_age_med"] = loo_median(
        x, ["행정동코드"], "age_m"
    )

    x["n_dong_age"] = (
        x.groupby("행정동코드")["open_s"]
        .transform("size") - 1
    )

    opened = d.loc[
        d["open_s"].ge(t6) & d["open_s"].lt(t)
    ]

    closed = d.loc[
        d["close_s"].ge(t6) & d["close_s"].lt(t)
    ]

    previous_active = d.loc[active(d, t6)]

    life = d.loc[
        d["close_s"].ge(t12) & d["close_s"].lt(t)
    ].copy()

    life["life_m"] = age_months(
        life["close_s"], life["open_s"]
    )

    cohort = d.loc[
        d["open_s"].ge(t12) & d["open_s"].lt(t6)
    ].copy()

    anniversary = (
        cohort["open_s"] + pd.DateOffset(months=6)
    )

    cohort["survived"] = (
        cohort["close_s"].isna()
        | cohort["close_s"].gt(anniversary)
    ).astype(float)

    for level, keys in [
        ("h", ["행정동코드", "업종명"]),
        ("g", ["구군", "업종명"]),
    ]:
        n = lookup(x, x, keys).fillna(0) - 1
        x[f"n_{level}k"] = n

        franchise_sum = (
            lookup(x, x, keys, "is_franchise", "sum")
            - x["is_franchise"]
        )

        franchise_n = (
            lookup(x, x, keys, "is_franchise", "count")
            - x["is_franchise"].notna().astype(int)
        )

        x[f"n_fr_{level}"] = franchise_n

        x[f"fr_share_{level}_raw"] = (
            franchise_sum / franchise_n.replace(0, np.nan)
        )

        denominator = (
            lookup(x, previous_active, keys).fillna(0)
            - active(x, t6).astype(int)
        )

        n_open = (
            lookup(x, opened, keys).fillna(0)
            - x["open_s"].ge(t6).astype(int)
        )

        n_close = lookup(x, closed, keys).fillna(0)

        x[f"n_start_{level}k"] = denominator
        x[f"n_open_{level}k"] = n_open
        x[f"n_close_{level}k"] = n_close

        x[f"close_rate_6m_{level}_raw"] = (
            n_close / denominator.replace(0, np.nan)
        )

        x[f"open_rate_6m_{level}_raw"] = (
            n_open / denominator.replace(0, np.nan)
        )

        x[f"net_6m_{level}_raw"] = n_open - n_close

        x[f"n_closed_life_{level}"] = (
            lookup(x, life, keys).fillna(0)
        )

        x[f"closed_life_avg_{level}_raw"] = lookup(
            x, life, keys, "life_m", "mean"
        )

        own_cohort = (
            x["open_s"].ge(t12)
            & x["open_s"].lt(t6)
        ).astype(int)

        cohort_n = (
            lookup(x, cohort, keys).fillna(0)
            - own_cohort
        )

        survivor_n = (
            lookup(x, cohort, keys, "survived", "sum").fillna(0)
            - own_cohort
        )

        x[f"n_cohort_{level}"] = cohort_n

        x[f"surv_6m_{level}_raw"] = (
            survivor_n / cohort_n.replace(0, np.nan)
        )

    # 표본 부족 보정
    for feature in [
        "fr_share",
        "close_rate_6m",
        "open_rate_6m",
        "net_6m",
    ]:
        dong_value = x[f"{feature}_h_raw"]
        gu_value = x[f"{feature}_g_raw"]

        corrected = (
            dong_value * x["n_hk"]
            + gu_value * PRIOR_WEIGHT
        ) / (x["n_hk"] + PRIOR_WEIGHT)

        x[feature] = dong_value.where(
            x["n_hk"].ge(SMALL_N),
            corrected,
        )

    x["closed_life_avg"] = x["closed_life_avg_h_raw"].where(
        x["n_closed_life_h"].ge(5),
        x["closed_life_avg_g_raw"],
    )

    x["surv_6m"] = x["surv_6m_h_raw"]

    # 업종 다양성: 자기 자신 제외
    counts = (
        x.groupby(["행정동코드", "업종명"])["open_s"]
        .transform("size")
        .astype(float)
    )

    total = (
        x.groupby("행정동코드")["open_s"]
        .transform("size")
        .astype(float)
    )

    terms = counts * np.log(counts)

    sum_terms = (
        (terms / counts)
        .groupby(x["행정동코드"])
        .transform("sum")
    )

    remaining = counts - 1

    remaining_term = (
        remaining * np.log(remaining.where(remaining.gt(0), 1))
    )

    x["biz_entropy"] = (
        np.log((total - 1).where(total.gt(1)))
        - (sum_terms - terms + remaining_term)
        / (total - 1).replace(0, np.nan)
    )

    x["n_entropy"] = total - 1

    if t < pd.Timestamp("2023-07-01"):
        x[["close_rate_6m", "open_rate_6m", "net_6m"]] = np.nan

    if t < pd.Timestamp("2024-01-01"):
        x[["closed_life_avg", "surv_6m"]] = np.nan

    if t12 < CLOSE_HISTORY_START:
        x["closed_life_avg"] = np.nan

    if t6 < CLOSE_HISTORY_START:
        x[["close_rate_6m", "net_6m"]] = np.nan

    # 향후 6개월 이내 폐업 여부
    label_end = t + pd.DateOffset(months=6)

    x["label_close_6m"] = pd.Series(
        pd.NA, index=x.index, dtype="Int8"
    )

    if (
        observed_through is not None
        and label_end - pd.Timedelta(days=1) <= observed_through
    ):
        x["label_close_6m"] = (
            x["close_s"].ge(t)
            & x["close_s"].lt(label_end)
        ).astype("Int8")

    return x

In [ ]:
def radians_coords(frame):
    return np.radians(
        frame[["lat", "lon"]].to_numpy(dtype=float)
    )


def calculate_spatial_features(snapshot, stores, t):
    x = snapshot.drop(
        columns=[
            "lon",
            "lat",
            "coord_valid",
            "same_300m",
            "close_100m_6m",
        ],
        errors="ignore",
    ).copy()

    x = x.merge(
        stores[
            ["점포이력ID", "lon", "lat", "coord_valid"]
        ],
        on="점포이력ID",
        how="left",
        validate="one_to_one",
    )

    x["coord_valid"] = (
        x["coord_valid"].fillna(False).astype(bool)
    )

    same_counts = pd.Series(
        np.nan, index=x.index, dtype=float
    )

    closed_counts = pd.Series(
        np.nan, index=x.index, dtype=float
    )

    valid_active = x.loc[x["coord_valid"]]

    # 300m 이내 동종 영업점 수: 자신 제외
    for _, group in valid_active.groupby(
        "업종명", sort=False, observed=True
    ):
        points = radians_coords(group)
        tree = BallTree(points, metric="haversine")

        counts = tree.query_radius(
            points,
            r=300 / EARTH_RADIUS_M,
            count_only=True,
        )

        same_counts.loc[group.index] = counts - 1

    # 100m 이내 최근 6개월 폐업 수: 모든 업종
    start = t - pd.DateOffset(months=6)

    if start >= CLOSE_HISTORY_START:
        recent_closed = stores.loc[
            stores["coord_valid"]
            & stores["close_s"].ge(start)
            & stores["close_s"].lt(t)
        ]

        if len(valid_active) > 0:
            if recent_closed.empty:
                closed_counts.loc[valid_active.index] = 0
            else:
                tree = BallTree(
                    radians_coords(recent_closed),
                    metric="haversine",
                )

                counts = tree.query_radius(
                    radians_coords(valid_active),
                    r=100 / EARTH_RADIUS_M,
                    count_only=True,
                )

                closed_counts.loc[valid_active.index] = counts

    x["same_300m"] = same_counts.astype("Int32")
    x["close_100m_6m"] = closed_counts.astype("Int32")

    return x

In [ ]:
extension_months = pd.date_range(
    "2026-05-01",
    "2026-08-01",
    freq="MS",
)

template_path = (
    OLD_SNAPSHOT_DIR / "snapshot_202604.parquet"
)

template_columns = pq.read_schema(
    template_path
).names

extension_reports = []

for i, t in enumerate(extension_months, start=1):
    print(
        f"[{i}/{len(extension_months)}] "
        f"{t:%Y-%m} 계산 시작",
        flush=True,
    )

    snapshot = build_month(
        d,
        t,
        OBSERVED_THROUGH,
    )

    updated = calculate_spatial_features(
        snapshot,
        stores,
        t,
    )

    if set(updated.columns) != set(template_columns):
        raise ValueError(
            f"{t:%Y-%m}: 기존 결과와 컬럼이 다릅니다.\n"
            f"누락: {set(template_columns) - set(updated.columns)}\n"
            f"추가: {set(updated.columns) - set(template_columns)}"
        )

    updated = updated[template_columns]

    assert len(updated) == len(snapshot)
    assert not updated["점포이력ID"].duplicated().any()
    assert updated["label_close_6m"].isna().all()

    path = (
        NEW_SNAPSHOT_DIR / f"snapshot_{t:%Y%m}.parquet"
    )

    temp_path = path.with_suffix(".parquet.part")

    updated.to_parquet(
        temp_path,
        index=False,
        compression="snappy",
    )

    saved_rows = pq.ParquetFile(
        temp_path
    ).metadata.num_rows

    if saved_rows != len(updated):
        raise ValueError("저장 행 수 불일치")

    temp_path.replace(path)

    extension_reports.append({
        "기준월": t.strftime("%Y-%m"),
        "행수": len(updated),
        "파일": str(path),
    })

    pd.DataFrame(extension_reports).to_csv(
        NEW_OUT / "추가기간_계산요약.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"→ {t:%Y-%m}: {len(updated):,}행 저장 완료",
        flush=True,
    )

    del snapshot, updated
    gc.collect()

display(pd.DataFrame(extension_reports))

In [ ]:
all_months = pd.date_range(
    "2023-01-01",
    "2026-08-01",
    freq="MS",
)

paths = [
    (
        OLD_SNAPSHOT_DIR
        if t <= pd.Timestamp("2026-04-01")
        else NEW_SNAPSHOT_DIR
    ) / f"snapshot_{t:%Y%m}.parquet"
    for t in all_months
]

missing = [
    str(path) for path in paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "누락된 파일:\n" + "\n".join(missing)
    )


def normalize_schema(schema):
    return pa.schema([
        pa.field(
            field.name,
            (
                field.type.value_type
                if pa.types.is_dictionary(field.type)
                else field.type
            ),
            nullable=True,
        )
        for field in schema
    ])


schemas = [
    normalize_schema(pq.read_schema(path))
    for path in paths
]

columns = schemas[0].names

for path, schema in zip(paths, schemas):
    if set(schema.names) != set(columns):
        raise ValueError(
            f"컬럼 구성 불일치: {path.name}"
        )

schemas = [
    pa.schema([schema.field(col) for col in columns])
    for schema in schemas
]

target_schema = pa.unify_schemas(
    schemas,
    promote_options="permissive",
)

label_index = target_schema.get_field_index(
    "label_close_6m"
)

if label_index < 0:
    raise ValueError("label_close_6m 컬럼이 없습니다.")

target_schema = target_schema.set(
    label_index,
    pa.field(
        "label_close_6m",
        pa.int8(),
        nullable=True,
    ),
)

output_path = NEW_OUT / (
    "인허가_파생지표_전체_202301_202608.parquet"
)

temp_path = output_path.with_suffix(".parquet.part")

total_rows = 0
reports = []

with pq.ParquetWriter(
    temp_path,
    target_schema,
    compression="snappy",
) as writer:

    for i, (t, path) in enumerate(
        zip(all_months, paths),
        start=1,
    ):
        source = pq.ParquetFile(path)

        expected_rows = source.metadata.num_rows
        month_rows = 0
        n_labels = 0
        n_positive = 0

        label_end = t + pd.DateOffset(months=6)

        fully_observed = (
            label_end
            <= OBSERVED_THROUGH + pd.Timedelta(days=1)
        )

        for batch in source.iter_batches(
            batch_size=BATCH_SIZE,
            use_threads=False,
        ):
            table = pa.Table.from_batches([batch])

            base_values = table["base_ym"].to_pandas()

            if not base_values.eq(
                t.strftime("%Y-%m")
            ).fillna(False).all():
                raise ValueError(
                    f"{path.name}: 기준월 불일치"
                )

            # 6개월 전체가 관측된 기준월만 정답 생성
            if fully_observed:
                close_dates = pd.to_datetime(
                    table["close_s"].to_pandas(),
                    errors="raise",
                )

                labels = (
                    close_dates.ge(t)
                    & close_dates.lt(label_end)
                ).astype("int8")

                label_array = pa.array(
                    labels,
                    type=pa.int8(),
                )

                n_labels += len(labels)
                n_positive += int(labels.sum())

                del close_dates, labels

            else:
                label_array = pa.nulls(
                    table.num_rows,
                    type=pa.int8(),
                )

            current_label_index = (
                table.schema.get_field_index(
                    "label_close_6m"
                )
            )

            table = table.set_column(
                current_label_index,
                "label_close_6m",
                label_array,
            )

            table = table.select(
                target_schema.names
            ).cast(target_schema)

            writer.write_table(table)

            month_rows += table.num_rows
            total_rows += table.num_rows

            del batch, table, label_array, base_values

        if month_rows != expected_rows:
            raise ValueError(
                f"{path.name}: 입력·출력 행 수 불일치"
            )

        reports.append({
            "기준월": t.strftime("%Y-%m"),
            "행수": month_rows,
            "정답있는행수": n_labels,
            "폐업정답행수": (
                n_positive if n_labels else np.nan
            ),
        })

        print(
            f"[{i:02d}/{len(paths)}] {t:%Y-%m} | "
            f"{month_rows:,}행 | "
            f"누적 {total_rows:,}행",
            flush=True,
        )

        del source
        gc.collect()

saved_rows = pq.ParquetFile(
    temp_path
).metadata.num_rows

if saved_rows != total_rows:
    raise ValueError("전체 행 수 검증 실패")

temp_path.replace(output_path)

summary = pd.DataFrame(reports)

summary.to_csv(
    NEW_OUT / "전체기간_저장요약.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n전체 파일 저장 완료")
print("파일:", output_path)
print(f"기간: {len(all_months)}개월")
print(f"전체 행 수: {saved_rows:,}")
print(f"컬럼 수: {len(target_schema.names)}")
print(
    f"크기: "
    f"{output_path.stat().st_size / 1024**2:,.1f} MB"
)

display(summary.tail(8))

In [ ]:
import re

name_map = {
    # 점포 기본 정보
    "점포이력ID": "점포번호",
    "base_ym": "기준월",
    "open_s": "개업일",
    "close_s": "폐업일",
    "lon": "경도",
    "lat": "위도",
    "coord_valid": "좌표유효여부",

    # 점포 특성
    "age_m": "영업기간_개월",
    "age_band": "영업기간_구간",
    "is_new": "개업1년미만",
    "biz_type": "점포업종",
    "is_franchise": "프랜차이즈여부",

    # 최종 파생지표
    "dong_age_med": "동네점포_영업기간중앙값_개월",
    "closed_life_avg": "주변동종_폐업전평균영업기간_개월",
    "surv_6m": "주변동종_신규점포_6개월생존율",
    "fr_share": "주변동종_프랜차이즈비율",
    "close_rate_6m": "주변동종_6개월폐업률",
    "open_rate_6m": "주변동종_6개월개업률",
    "net_6m": "주변동종_6개월점포증감",
    "biz_entropy": "동네_업종다양성",

    # 공간 지표
    "same_300m": "300미터내_동종점포수",
    "close_100m_6m": "100미터내_6개월폐업수",

    # 집계에 사용한 점포 수
    "n_hk": "동네_동종점포수",
    "n_gk": "구군_동종점포수",
    "n_dong_age": "동네_영업기간계산점포수",
    "n_entropy": "동네_다양성계산점포수",

    # 정답
    "label_close_6m": "앞으로6개월내_폐업여부",
}

# 동네·구군별 원래 계산값과 표본 수
for code, region in [
    ("h", "동네"),
    ("g", "구군"),
]:
    name_map.update({
        f"fr_share_{code}_raw":
            f"{region}동종_프랜차이즈비율_원래값",

        f"close_rate_6m_{code}_raw":
            f"{region}동종_6개월폐업률_원래값",

        f"open_rate_6m_{code}_raw":
            f"{region}동종_6개월개업률_원래값",

        f"net_6m_{code}_raw":
            f"{region}동종_6개월점포증감_원래값",

        f"closed_life_avg_{code}_raw":
            f"{region}동종_폐업전평균영업기간_개월_원래값",

        f"surv_6m_{code}_raw":
            f"{region}동종_신규점포_6개월생존율_원래값",

        f"n_fr_{code}":
            f"{region}동종_프랜차이즈확인점포수",

        f"n_start_{code}k":
            f"{region}동종_6개월전점포수",

        f"n_open_{code}k":
            f"{region}동종_6개월개업수",

        f"n_close_{code}k":
            f"{region}동종_6개월폐업수",

        f"n_closed_life_{code}":
            f"{region}동종_1년폐업수",

        f"n_cohort_{code}":
            f"{region}동종_생존율계산점포수",
    })

old_columns = df_preview.columns.tolist()

new_columns = [
    name_map.get(col, col)
    for col in old_columns
]

# 누락된 영문 이름과 중복 확인
unmapped = [
    col for col in new_columns
    if re.search(r"[A-Za-z]", col)
]

if unmapped:
    raise ValueError(
        f"한글 이름을 추가로 지정해야 하는 컬럼: {unmapped}"
    )

if len(new_columns) != len(set(new_columns)):
    raise ValueError("변경 후 컬럼명이 중복됩니다.")

df_preview.columns = new_columns

print("컬럼명 변경 완료")

with pd.option_context("display.max_columns", None):
    display(df_preview.head(10))

print("\n전체 컬럼명")
print(df_preview.columns.tolist())

In [ ]:
from pathlib import Path
import re
import gc
import pyarrow as pa
import pyarrow.parquet as pq

# 기존 전체 파일
source_path = Path(output_path)

# 한글 컬럼으로 저장할 새 파일
korean_path = source_path.with_name(
    source_path.stem + "_한글컬럼.parquet"
)

temp_path = korean_path.with_suffix(".parquet.part")

source = pq.ParquetFile(source_path)
old_schema = source.schema_arrow

new_columns = [
    name_map.get(col, col)
    for col in old_schema.names
]

unmapped = [
    col for col in new_columns
    if re.search(r"[A-Za-z]", col)
]

if unmapped:
    raise ValueError(
        f"한글 이름을 추가로 지정해야 하는 컬럼: {unmapped}"
    )

if len(new_columns) != len(set(new_columns)):
    raise ValueError("변경 후 컬럼명이 중복됩니다.")

# 기존 자료형 유지, 이름만 변경
new_schema = pa.schema([
    field.with_name(new_name)
    for field, new_name in zip(old_schema, new_columns)
])

total_rows = 0
expected_rows = source.metadata.num_rows

with pq.ParquetWriter(
    temp_path,
    new_schema,
    compression="snappy",
) as writer:

    for i, batch in enumerate(
        source.iter_batches(
            batch_size=20_000,
            use_threads=False,
        ),
        start=1,
    ):
        table = pa.Table.from_batches([batch])

        # 이전 영문 컬럼명을 담은 pandas 메타데이터 제거
        table = (
            table.rename_columns(new_columns)
            .replace_schema_metadata(None)
            .cast(new_schema)
        )

        writer.write_table(table)
        total_rows += table.num_rows

        del table, batch

        if i % 20 == 0:
            print(
                f"{total_rows:,} / {expected_rows:,}행 처리",
                flush=True,
            )
            gc.collect()

saved_rows = pq.ParquetFile(temp_path).metadata.num_rows

if not (saved_rows == total_rows == expected_rows):
    raise ValueError("저장 행 수가 원본과 다릅니다.")

temp_path.replace(korean_path)

del source
gc.collect()

print("\n전체 파일 한글 컬럼 변경 완료")
print("파일:", korean_path)
print(f"전체 행 수: {saved_rows:,}")

In [ ]:
source = pq.ParquetFile(korean_path)

print("전체 컬럼명:")
print(source.schema_arrow.names)

batch = next(
    source.iter_batches(
        batch_size=1_000,
        use_threads=False,
    )
)

df_preview = batch.to_pandas()

with pd.option_context("display.max_columns", None):
    display(df_preview.head(10))

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display

import unicodedata
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.dataset as ds

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

filename = (
    "인허가_파생지표_전체_202301_202608_한글컬럼.parquet"
)

matches = [
    path
    for path in ROOT.rglob("*.parquet")
    if unicodedata.normalize("NFC", path.name) == filename
]

if len(matches) != 1:
    raise ValueError(
        f"파일이 {len(matches)}개 발견됐습니다:\n"
        + "\n".join(map(str, matches))
    )

parquet_path = matches[0]
print("사용 파일:", parquet_path)

In [ ]:
source = pq.ParquetFile(parquet_path)

print(f"전체 행 수: {source.metadata.num_rows:,}")
print(f"전체 컬럼 수: {len(source.schema_arrow.names):,}")

display(
    pd.DataFrame({
        "컬럼명": source.schema_arrow.names,
        "자료형": [
            str(field.type)
            for field in source.schema_arrow
        ],
    })
)

In [ ]:
MONTH = "2026-08"  # 예: "2026-02", "2025-08"
PREVIEW_ROWS = 1_000

dataset = ds.dataset(
    parquet_path,
    format="parquet",
)

scanner = dataset.scanner(
    filter=ds.field("기준월") == MONTH,
    batch_size=PREVIEW_ROWS,
    batch_readahead=1,
    fragment_readahead=1,
    use_threads=False,
)

df_preview = None

for batch in scanner.to_batches():
    if batch.num_rows > 0:
        df_preview = batch.slice(
            0, PREVIEW_ROWS
        ).to_pandas()
        break

if df_preview is None:
    raise ValueError(f"{MONTH} 기준월 데이터가 없습니다.")

print("확인 기준월:", MONTH)
print("미리보기 크기:", df_preview.shape)

with pd.option_context("display.max_columns", None):
    display(df_preview.head(10))

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

schema = pq.read_schema(parquet_path)
columns = schema.names

# 1. 같은 업태 점포 수 컬럼 확인
expected_type_cols = [
    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]

type_check = pd.DataFrame({
    "컬럼명": expected_type_cols,
    "파일에_포함": [
        col in columns
        for col in expected_type_cols
    ],
})

print("① 같은 업태 점포 수")
display(type_check)

# 혹시 다른 이름으로 저장되어 있는지도 확인
related_cols = [
    col for col in columns
    if "업태" in col
]

print("업태 관련 전체 컬럼:", related_cols)


# 2. 업종별 0·1 컬럼 확인
industry_cols = [
    col for col in columns
    if col.startswith("업종__")
]

print(f"\n② 업종별 0·1 컬럼: {len(industry_cols)}개")

if industry_cols:
    display(pd.DataFrame({
        "컬럼명": industry_cols,
        "자료형": [
            str(schema.field(col).type)
            for col in industry_cols
        ],
    }))
else:
    print("이 파일에는 업종__로 시작하는 컬럼이 없습니다.")


# 3. 확인 결과
print("\n③ 결과")

if all(col in columns for col in expected_type_cols):
    print("✅ 같은 업태 점포 수 3개 모두 포함")
else:
    missing = [
        col for col in expected_type_cols
        if col not in columns
    ]
    print("❌ 누락된 업태 점포 수 컬럼:", missing)

if industry_cols:
    print(f"✅ 업종별 0·1 컬럼 {len(industry_cols)}개 포함")
else:
    print("❌ 업종별 0·1 컬럼 미포함")

In [ ]:
!pip -q install pyarrow scikit-learn

from google.colab import drive
from pathlib import Path
from IPython.display import display

import gc
import unicodedata
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

from sklearn.neighbors import BallTree

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/KB_FinalProject_Team6"
)

BATCH_SIZE = 20_000
EARTH_RADIUS_M = 6_371_008.8


def find_unique(filename):
    suffix = Path(filename).suffix

    matches = [
        path for path in ROOT.rglob(f"*{suffix}")
        if unicodedata.normalize("NFC", path.name) == filename
    ]

    if len(matches) != 1:
        raise ValueError(
            f"{filename}: {len(matches)}개 발견\n"
            + "\n".join(map(str, matches))
        )

    return matches[0]


SOURCE_PATH = find_unique(
    "인허가_파생지표_전체_202301_202608_한글컬럼.parquet"
)

MASTER_PATH = find_unique("인허가_master.csv")

OUTPUT_PATH = SOURCE_PATH.with_name(
    SOURCE_PATH.stem + "_업태업종추가.parquet"
)

TEMP_PATH = OUTPUT_PATH.with_suffix(".parquet.part")

print("기존 전체 파일:", SOURCE_PATH)
print("인허가 원본:", MASTER_PATH)
print("새 저장 파일:", OUTPUT_PATH)

In [ ]:
for encoding in ["utf-8-sig", "cp949"]:
    try:
        header = pd.read_csv(
            MASTER_PATH,
            encoding=encoding,
            nrows=0,
        )
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("원본 CSV 인코딩을 확인하세요.")

required = ["점포이력ID", "업태구분명"]
missing = set(required) - set(header.columns)

if missing:
    raise ValueError(f"원본 필수 컬럼 누락: {sorted(missing)}")

industry_cols = [
    col for col in header.columns
    if col.startswith("업종__")
]

if not industry_cols:
    raise ValueError("업종__로 시작하는 컬럼이 없습니다.")

read_cols = required + industry_cols

raw_attributes = pd.read_csv(
    MASTER_PATH,
    encoding=encoding,
    usecols=read_cols,
    dtype={
        col: ("Int8" if col in industry_cols else "string")
        for col in read_cols
    },
)

if (
    raw_attributes["점포이력ID"].isna().any()
    or raw_attributes["점포이력ID"].duplicated().any()
):
    raise ValueError("원본 점포이력ID의 결측·중복을 확인하세요.")

# 업태명의 앞뒤 공백만 정리
raw_attributes["업태구분명"] = (
    raw_attributes["업태구분명"]
    .str.strip()
    .replace("", pd.NA)
)

for col in industry_cols:
    if (
        raw_attributes[col].isna().any()
        or not raw_attributes[col].isin([0, 1]).all()
    ):
        raise ValueError(
            f"{col}: 결측 또는 0·1 이외 값이 있습니다."
        )

    raw_attributes[col] = raw_attributes[col].astype("int8")

attributes = (
    raw_attributes
    .rename(columns={"점포이력ID": "점포번호"})
    .set_index("점포번호")
)

del raw_attributes
gc.collect()

print(f"원본 점포 수: {len(attributes):,}")
print(f"복원할 업종별 0·1 컬럼: {len(industry_cols)}개")
print(
    "업태 결측 점포:",
    f"{attributes['업태구분명'].isna().sum():,}개",
)
print(
    "연결표 메모리:",
    f"{attributes.memory_usage(deep=True).sum() / 1024**2:,.1f} MB",
)

In [ ]:
source_info = pq.ParquetFile(SOURCE_PATH)

source_schema = source_info.schema_arrow.remove_metadata()
source_columns = source_schema.names
original_total_rows = source_info.metadata.num_rows

# 동종 기준으로 사용할 대표 업종
if "업종명" in source_columns:
    BIZ_COL = "업종명"
elif "점포업종" in source_columns:
    BIZ_COL = "점포업종"
else:
    raise ValueError("업종명 또는 점포업종 컬럼이 필요합니다.")

required = [
    "기준월",
    "점포번호",
    "행정동코드",
    "구군",
    BIZ_COL,
    "경도",
    "위도",
]

missing = set(required) - set(source_columns)

if missing:
    raise ValueError(f"전체 파일 필수 컬럼 누락: {sorted(missing)}")

COUNT_COLS = [
    "동네_같은업태점포수",
    "구군_같은업태점포수",
    "300미터내_같은업태점포수",
]

# 재실행해도 중복되지 않도록 추가 대상 컬럼은 원본에서 다시 연결
refresh_cols = (
    {"업태구분명"}
    | set(COUNT_COLS)
    | set(industry_cols)
)

base_cols = [
    col for col in source_columns
    if col not in refresh_cols
]

# 출력 자료형: 기존 자료형은 유지
output_schema = pa.schema(
    [source_schema.field(col) for col in base_cols]
    + [pa.field("업태구분명", pa.string())]
    + [pa.field(col, pa.int32()) for col in COUNT_COLS]
    + [pa.field(col, pa.int8()) for col in industry_cols]
)

month_set = set()

for batch in source_info.iter_batches(
    batch_size=BATCH_SIZE,
    columns=["기준월"],
    use_threads=False,
):
    month_set.update(batch.column(0).to_pylist())

if None in month_set:
    raise ValueError("기준월 결측이 있습니다.")

months = sorted(month_set)

# 기준월 형식 검증
for month in months:
    parsed = pd.to_datetime(month, format="%Y-%m")
    if parsed.strftime("%Y-%m") != month:
        raise ValueError(f"기준월 형식 확인 필요: {month}")

dataset = ds.dataset(
    SOURCE_PATH,
    format="parquet",
)

del source_info
gc.collect()

print(f"전체 행 수: {original_total_rows:,}")
print(f"기준월: {months[0]} ~ {months[-1]}")
print(f"처리할 기준월: {len(months)}개")
print(f"완료 후 컬럼 수: {len(output_schema.names)}개")

In [ ]:
def calculate_type_counts(month_frame):
    month_frame = month_frame.reset_index(drop=True).copy()

    month_frame["점포번호"] = (
        month_frame["점포번호"].astype("string")
    )

    if (
        month_frame["점포번호"].isna().any()
        or month_frame["점포번호"].duplicated().any()
    ):
        raise ValueError("해당 월의 점포번호에 결측·중복이 있습니다.")

    ids = month_frame["점포번호"]

    if not ids.isin(attributes.index).all():
        raise ValueError(
            "원본 인허가 정보와 연결되지 않는 점포가 있습니다."
        )

    month_frame["업태구분명"] = (
        attributes["업태구분명"].reindex(ids).array
    )

    month_frame[BIZ_COL] = (
        month_frame[BIZ_COL]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    valid_type = (
        month_frame[BIZ_COL].notna()
        & month_frame["업태구분명"].notna()
    )

    for col in COUNT_COLS:
        month_frame[col] = pd.Series(
            pd.NA,
            index=month_frame.index,
            dtype="Int32",
        )

    # 같은 행정동·업종·업태, 자신 제외
    for column, keys in [
        (
            "동네_같은업태점포수",
            ["행정동코드", BIZ_COL, "업태구분명"],
        ),
        (
            "구군_같은업태점포수",
            ["구군", BIZ_COL, "업태구분명"],
        ),
    ]:
        counts = (
            month_frame.loc[valid_type]
            .groupby(keys, observed=True)["점포번호"]
            .transform("size")
            - 1
        )

        month_frame.loc[
            counts.index, column
        ] = counts.astype("Int32")

    # 좌표 검사
    for col in ["경도", "위도"]:
        month_frame[col] = pd.to_numeric(
            month_frame[col],
            errors="coerce",
        ).astype(float)

    valid_coord = (
        np.isfinite(month_frame["경도"])
        & np.isfinite(month_frame["위도"])
        & month_frame["경도"].between(-180, 180)
        & month_frame["위도"].between(-90, 90)
        & ~(
            month_frame["경도"].eq(0)
            & month_frame["위도"].eq(0)
        )
    )

    spatial_ok = valid_type & valid_coord

    # 같은 업종·업태 중 반경 300m 이내, 자신 제외
    for _, group in month_frame.loc[spatial_ok].groupby(
        [BIZ_COL, "업태구분명"],
        observed=True,
        sort=False,
    ):
        points = np.radians(
            group[["위도", "경도"]].to_numpy(dtype=float)
        )

        tree = BallTree(
            points,
            metric="haversine",
        )

        counts = tree.query_radius(
            points,
            r=300 / EARTH_RADIUS_M,
            count_only=True,
        ) - 1

        month_frame.loc[
            group.index,
            "300미터내_같은업태점포수",
        ] = pd.array(counts, dtype="Int32")

    for col in COUNT_COLS:
        assert month_frame[col].dropna().ge(0).all()

    quality = {
        "행수": len(month_frame),
        "업태결측행수": int(
            month_frame["업태구분명"].isna().sum()
        ),
        "좌표결측또는오류행수": int((~valid_coord).sum()),
    }

    result = month_frame.set_index("점포번호")[
        ["업태구분명"] + COUNT_COLS
    ]

    return result, quality

In [ ]:
projection_cols = list(dict.fromkeys([
    "점포번호",
    "행정동코드",
    "구군",
    BIZ_COL,
    "경도",
    "위도",
]))

reports = []
total_rows = 0

with pq.ParquetWriter(
    TEMP_PATH,
    output_schema,
    compression="snappy",
) as writer:

    for month_number, month in enumerate(months, start=1):
        print(
            f"\n[{month_number:02d}/{len(months)}] "
            f"{month} 업태 집계 시작",
            flush=True,
        )

        month_filter = ds.field("기준월") == month

        # 해당 월의 집계용 컬럼만 메모리에 읽기
        month_frame = dataset.scanner(
            columns=projection_cols,
            filter=month_filter,
            batch_size=BATCH_SIZE,
            batch_readahead=1,
            fragment_readahead=1,
            use_threads=False,
        ).to_table().to_pandas()

        month_features, quality = calculate_type_counts(
            month_frame
        )

        expected_month_rows = len(month_frame)

        del month_frame
        gc.collect()

        month_rows = 0

        # 기존 전체 컬럼은 작은 배치로만 읽기
        scanner = dataset.scanner(
            columns=base_cols,
            filter=month_filter,
            batch_size=BATCH_SIZE,
            batch_readahead=1,
            fragment_readahead=1,
            use_threads=False,
        )

        for batch_number, batch in enumerate(
            scanner.to_batches(),
            start=1,
        ):
            if batch.num_rows == 0:
                continue

            table = (
                pa.Table.from_batches([batch])
                .replace_schema_metadata(None)
            )

            ids = table["점포번호"].to_pandas().astype("string")

            if not ids.isin(month_features.index).all():
                raise ValueError("월별 업태 집계 연결 실패")

            additions = month_features.reindex(ids)
            linked_attributes = attributes.reindex(ids)

            # 업태명 추가
            table = table.append_column(
                "업태구분명",
                pa.array(
                    additions["업태구분명"],
                    type=pa.string(),
                    from_pandas=True,
                ),
            )

            # 같은 업태 점포 수 3개 추가
            for col in COUNT_COLS:
                table = table.append_column(
                    col,
                    pa.array(
                        additions[col],
                        type=pa.int32(),
                        from_pandas=True,
                    ),
                )

            # 원본 업종별 0·1 컬럼 복원
            for col in industry_cols:
                table = table.append_column(
                    col,
                    pa.array(
                        linked_attributes[col],
                        type=pa.int8(),
                        from_pandas=True,
                    ),
                )

            table = table.cast(
                output_schema,
                safe=True,
            )

            writer.write_table(table)

            month_rows += table.num_rows
            total_rows += table.num_rows

            del (
                table,
                ids,
                additions,
                linked_attributes,
                batch,
            )

            if batch_number % 5 == 0:
                gc.collect()

        if month_rows != expected_month_rows:
            raise ValueError(
                f"{month}: 입력·출력 행 수가 다릅니다."
            )

        reports.append({
            "기준월": month,
            "저장행수": month_rows,
            "업태결측행수": quality["업태결측행수"],
            "좌표결측또는오류행수": (
                quality["좌표결측또는오류행수"]
            ),
        })

        pd.DataFrame(reports).to_csv(
            OUTPUT_PATH.with_suffix(".처리요약.csv"),
            index=False,
            encoding="utf-8-sig",
        )

        print(
            f"→ {month}: {month_rows:,}행 저장 완료 "
            f"| 누적 {total_rows:,}행",
            flush=True,
        )

        del month_features, scanner
        gc.collect()

# 최종 파일 행 수 검증
saved_rows = pq.ParquetFile(TEMP_PATH).metadata.num_rows

if not (
    saved_rows == total_rows == original_total_rows
):
    raise ValueError("전체 입력·출력 행 수가 다릅니다.")

TEMP_PATH.replace(OUTPUT_PATH)

print("\n전체 작업 완료")
print("새 파일:", OUTPUT_PATH)
print(f"전체 행 수: {saved_rows:,}")
print(f"컬럼 수: {len(output_schema.names):,}")
print(
    f"파일 크기: "
    f"{OUTPUT_PATH.stat().st_size / 1024**2:,.1f} MB"
)

display(pd.DataFrame(reports))

In [ ]:
saved_schema = pq.read_schema(OUTPUT_PATH)
saved_columns = saved_schema.names

print("같은 업태 점포 수 확인")
display(pd.DataFrame({
    "컬럼명": COUNT_COLS,
    "포함여부": [
        col in saved_columns
        for col in COUNT_COLS
    ],
}))

saved_industries = [
    col for col in saved_columns
    if col.startswith("업종__")
]

print(f"업종별 0·1 컬럼: {len(saved_industries)}개")

# 기존 변수도 유지됐는지 확인
assert set(base_cols).issubset(saved_columns)
assert set(COUNT_COLS).issubset(saved_columns)
assert set(industry_cols).issubset(saved_columns)

new_dataset = ds.dataset(
    OUTPUT_PATH,
    format="parquet",
)

preview_columns = list(dict.fromkeys([
    "기준월",
    "점포번호",
    BIZ_COL,
    "업태구분명",
    *COUNT_COLS,
    *industry_cols[:5],
]))

scanner = new_dataset.scanner(
    columns=preview_columns,
    filter=ds.field("기준월") == months[-1],
    batch_size=1_000,
    batch_readahead=1,
    fragment_readahead=1,
    use_threads=False,
)

for batch in scanner.to_batches():
    if batch.num_rows:
        df_preview = batch.to_pandas()
        display(df_preview.head(20))
        break

# 이후 조회 시 새 파일을 사용하도록 경로 갱신
parquet_path = OUTPUT_PATH

print("이후 사용할 파일:", parquet_path)

In [ ]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/KB_FinalProject_Team6/완전쓰는거_ver2/인허가_파생지표_202301_202608/인허가_파생지표_전체마지막.parquet")

In [ ]:
df.tail()

In [ ]:
df['기준월']

In [ ]:
df['카드업종']